# GoalPruningCodeAgent on GAIA

Same tools, dataset, scorer and eval loop as `naiveReAct.ipynb`, so the runs are directly
comparable. The only difference is the agent: `GoalPruningCodeAgent` states an expectation per
step, judges whether the observation met it, and drops the failed attempts before a satisfied step.

The headline metric is **tokens**, measurable in one run. Accuracy is a non-inferiority check only
-- the run-to-run noise floor on GAIA is about ±2.4pp, so a single pair of runs cannot resolve less
than that.

`JUDGE_ON = False` gives the prompt-only control: identical system prompt, no judge call, nothing
ever pruned. That arm separates "the Expectation line changed the model's behaviour" from
"pruning changed it".

In [ ]:
import os

# ===================== experiment configuration =====================
# Each has an env override so one notebook can drive several parallel sweeps.
MODEL = os.getenv("SMOLAGENTS_MODEL", "gpt-4o")
N_QUESTIONS = int(os.getenv("SMOLAGENTS_N_QUESTIONS", "165"))
MAX_STEPS = int(os.getenv("SMOLAGENTS_MAX_STEPS", "50"))
COMPLETION_THRESHOLD = float(os.getenv("SMOLAGENTS_COMPLETION_THRESHOLD", "0.5"))
JUDGE_ON = os.getenv("SMOLAGENTS_JUDGE_ON", "1") == "1"  # 0 = prompt-only control, no pruning
RUN_TAG = os.getenv("SMOLAGENTS_RUN_TAG", "_gp1")
SEARCH_RPS = float(os.getenv("SMOLAGENTS_SEARCH_RPS", "0.3"))
# ====================================================================

MODELS = {
    "gpt-4o": "openai",
    "gpt-5.4-mini": "openai",
    "Qwen/Qwen3.5-9B": "together",
    "Qwen/Qwen3.7-Plus": "together",
}
assert MODEL in MODELS, f"unknown MODEL {MODEL!r}; pick one of {list(MODELS)}"

RUN_NAME = f"goalprune{RUN_TAG}{'' if JUDGE_ON else '_nojudge'}_react_{MODEL}"

# build_tools reads both of these, so they must be set before it is called.
os.environ["SMOLAGENTS_DOWNLOADS_DIR"] = f"downloads_{RUN_NAME.replace('/', '_')}"
os.environ["SMOLAGENTS_SEARCH_RPS"] = str(SEARCH_RPS)

print(f"run   : {RUN_NAME}")
print(f"judge : {'on' if JUDGE_ON else 'OFF (control)'} | threshold={COMPLETION_THRESHOLD}")

In [ ]:
import sys

sys.path.insert(0, "../examples/open_deep_research")

from dotenv import load_dotenv

load_dotenv()

from smolagents import OpenAIModel

# Only Qwen3.7-Plus rejects non-streaming. Streaming loses the judge's logprobs (the agglomerated
# ChatMessage carries no raw response), so judge_completion falls back to reading the word
# "yes"/"no" -- a binary verdict instead of a graded one. Stream only where forced.
STREAM_ONLY = {"Qwen/Qwen3.7-Plus"}

if MODELS[MODEL] == "openai":
    model = OpenAIModel(model_id=MODEL, api_key=os.environ["OPENAI_API_KEY"])
else:
    # Together needs enable_thinking nested in chat_template_kwargs for vLLM-served models.
    model = OpenAIModel(
        model_id=MODEL,
        api_base="https://api.together.ai/v1/",
        api_key=os.environ["TOGETHER_API_KEY"],
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        client_kwargs={"timeout": 300.0},
    )
stream_outputs = MODEL in STREAM_ONLY
print(f"{MODEL} via {MODELS[MODEL]} | stream_outputs={stream_outputs}")

In [ ]:
# Standard GAIA tool stack, same as naiveReAct.ipynb.
from common_setup import build_tools

tools, ti_tool, visualizer = build_tools(model)

In [ ]:
from smolagents.context_pruning.goal_agent import GoalPruningCodeAgent
from smolagents.monitoring import LogLevel

agent = GoalPruningCodeAgent(
    tools=tools,
    model=model,
    max_steps=MAX_STEPS,
    verbosity_level=LogLevel.DEBUG,
    additional_authorized_imports=["pandas", "numpy", "PIL", "json", "io", "zipfile", "csv", "openpyxl"],
    completion_threshold=COMPLETION_THRESHOLD,
    judge_completion_calls=JUDGE_ON,
    stream_outputs=stream_outputs,
)

# The Expectation protocol only exists in this agent's own prompt, so assert it is the one in use.
assert "Expectation:" in agent.system_prompt, "system prompt is missing the Expectation protocol"
print(f"system prompt: {len(agent.system_prompt)} chars, {agent.system_prompt.count('Expectation:')} Expectation mentions")

In [ ]:
# Load GAIA validation set from HuggingFace
import pandas as pd
from common_setup import load_gaia_dataset

eval_ds = load_gaia_dataset(set_to_run="validation")
print(f"Loaded {len(eval_ds)} examples")

In [ ]:
from common_setup import evaluate_agent

results = evaluate_agent(
    agent,
    eval_ds,
    ti_tool,
    visualizer,
    n_samples=N_QUESTIONS,
    output_file=f"{RUN_NAME}.jsonl",
    pickle_dir=RUN_NAME,
)

In [ ]:
df = pd.DataFrame(results)
total = len(df)
correct = df["is_correct"].sum()

print("=== GAIA Evaluation Results ===")
print(f"Overall accuracy:      {correct}/{total} = {correct / total:.1%}")
print(f"Avg time per question: {df['time_taken_seconds'].mean():.1f}s")
print(f"Avg steps per question: {df['num_steps'].mean():.1f}")

tokens = df["token_counts"].apply(lambda x: x.get("total_tokens", 0))
print(f"Tokens per question:   mean {tokens.mean():,.0f} | median {tokens.median():,.0f}")

print("\nAccuracy by level:")
for level in sorted(df["task"].unique()):
    sub = df[df["task"] == level]
    print(f"  Level {level}: {sub['is_correct'].sum()}/{len(sub)} = {sub['is_correct'].mean():.1%}")

## Pruning diagnostics

A judged success is the only thing that prunes, so the commit rate *is* the mechanism. Near 0% and
nothing is pruned and the agent is plain ReAct paying for a judge call every step; near 100% and
every step closes its own subtask, which also prunes nothing since there is nothing earlier to
drop. Both degenerate cases mean nothing else about the design matters.

In [ ]:
import numpy as np

steps = [s for r in results for s in r["steps"]]
probs = [s.get("completion_prob") for s in steps]
scored = [p for p in probs if p is not None]
with_expectation = [s for s in steps if s.get("expectation")]

print(f"steps total            : {len(steps)}")
print(f"  stated an Expectation: {len(with_expectation)} ({len(with_expectation) / max(len(steps), 1):.1%}) <- prompt compliance")
print(f"  judge returned a prob: {len(scored)} ({len(scored) / max(len(steps), 1):.1%})")
print(f"  unparseable / skipped: {len(steps) - len(scored)}")

if scored:
    committed = [p for p in scored if p >= COMPLETION_THRESHOLD]
    print(f"\ncommit rate at p>={COMPLETION_THRESHOLD}: {len(committed)}/{len(scored)} = {len(committed) / len(scored):.1%}")
    print(f"probability distribution: {np.percentile(scored, [0, 10, 25, 50, 75, 90, 100]).round(3).tolist()}")
    print(f"  p<0.01: {sum(p < 0.01 for p in scored)} | 0.01-0.99: {sum(0.01 <= p <= 0.99 for p in scored)} | p>0.99: {sum(p > 0.99 for p in scored)}")

    # The judge is saturated in practice, so sweeping the threshold usually moves nothing --
    # which is worth knowing before tuning it.
    print("\ncommit rate vs threshold:")
    for t in (0.1, 0.3, 0.5, 0.7, 0.9):
        print(f"  {t:>4}: {sum(p >= t for p in scored) / len(scored):.1%}")

In [ ]:
# How much context did pruning actually remove? Replay the buffer offline over each
# trajectory's recorded probabilities, so the kept/dropped split is reconstructed exactly.
# A judged success is the only thing that closes a subtask, so this is a one-liner: everything
# before a commit is dropped, everything after the last commit is still open.
def visible_at_end(step_dicts, threshold):
    commits = [i for i, s in enumerate(step_dicts) if (s.get("completion_prob") or -1) >= threshold]
    return len(commits) + (len(step_dicts) - 1 - commits[-1] if commits else len(step_dicts))


rows = [
    {"n_steps": len(r["steps"]), "visible": visible_at_end(r["steps"], COMPLETION_THRESHOLD)}
    for r in results
    if r["steps"]
]
pr = pd.DataFrame(rows)
if len(pr):
    pr["retained"] = pr["visible"] / pr["n_steps"]
    print(f"steps still in the prompt at the end of a run: mean {pr['retained'].mean():.1%} | median {pr['retained'].median():.1%}")
    print(f"  questions where nothing was pruned at all: {(pr['retained'] >= 1.0).sum()}/{len(pr)} <- the plain-ReAct floor")
    print(pr.describe().round(2))

    # Retention under other thresholds, since the commit decision is recorded per step.
    print("\nretention vs threshold:")
    for t in (0.1, 0.3, 0.5, 0.7, 0.9):
        ret = np.mean([visible_at_end(r["steps"], t) / len(r["steps"]) for r in results if r["steps"]])
        print(f"  {t:>4}: {ret:.1%} of steps retained")

## Paired comparison against the naive baseline

Restricted to questions both runs completed. Tokens are compared on the **median** as well as the
mean: per-question totals are tail-dominated, so a mean ratio can be moved by one runaway question.

In [ ]:
import pickle
from pathlib import Path

BASELINE_DIR = Path(f"naive_react_{MODEL}_False_fixedbrowser")


def load_dir(d):
    out = {}
    for f in sorted(Path(d).glob("*.pkl")):
        try:
            with open(f, "rb") as fh:
                out[int(f.stem)] = pickle.load(fh)
        except Exception as e:
            print(f"  skipped {f}: {e}")
    return out


base = load_dir(BASELINE_DIR)
gp = load_dir(RUN_NAME)
shared = sorted(set(base) & set(gp))
print(f"baseline {len(base)} | goal-pruning {len(gp)} | paired {len(shared)}")

if shared:
    def tok(r):
        return r["token_counts"].get("total_tokens", 0)

    b_tok = np.array([tok(base[i]) for i in shared], dtype=float)
    g_tok = np.array([tok(gp[i]) for i in shared], dtype=float)
    b_steps = np.array([base[i]["num_steps"] for i in shared], dtype=float)
    g_steps = np.array([gp[i]["num_steps"] for i in shared], dtype=float)
    b_ok = np.array([bool(base[i]["is_correct"]) for i in shared])
    g_ok = np.array([bool(gp[i]["is_correct"]) for i in shared])

    print(f"\naccuracy : naive {b_ok.mean():.1%} -> goal-prune {g_ok.mean():.1%}")
    print(f"  flipped : {int((~b_ok & g_ok).sum())} gained, {int((b_ok & ~g_ok).sum())} lost")
    print(f"tokens   : median {b_tok.mean() and np.median(b_tok):,.0f} -> {np.median(g_tok):,.0f} ({np.median(g_tok) / np.median(b_tok):.2f}x)")
    print(f"           total  {b_tok.sum():,.0f} -> {g_tok.sum():,.0f} ({g_tok.sum() / b_tok.sum():.2f}x)")
    print(f"steps    : mean {b_steps.mean():.1f} -> {g_steps.mean():.1f}")

    # Exact two-sided McNemar over the discordant pairs, same as notes.ipynb (no statsmodels here).
    from math import comb

    x, y = int((b_ok & ~g_ok).sum()), int((~b_ok & g_ok).sum())
    p = 1.0 if x + y == 0 else min(2 * sum(comb(x + y, k) for k in range(min(x, y) + 1)) / 2 ** (x + y), 1.0)
    print(f"\nMcNemar exact p = {p:.3f}  (naive only {x}, goal-prune only {y})")